In [3]:
# -*- coding: utf-8 -*-
# --- Plot por ALGORITMO (agrega múltiplas pastas/execuções do mesmo alg) ---

import os
import re
import glob
from collections import defaultdict, Counter
from typing import Optional, Set, Dict, List

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.python.summary.summary_iterator import summary_iterator
from scipy.ndimage import gaussian_filter1d

# ---------- leitura de eventos ----------
def get_event_files_in_dir(d: str) -> List[str]:
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))

def safe_iter_events(path: str):
    try:
        for e in summary_iterator(path):
            yield e
    except Exception as ex:
        print(f"⚠️  Ignorando {path} (erro de leitura: {ex})")

# ---------- utilitários de tags ----------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")
def strip_ray_prefix(tag: str) -> str:
    # remove "ray/tune/" ou "ray/train/" etc. para facilitar matching
    return RAY_PREFIX.sub("", tag)

def tag_frequency(event_files, max_per_file=500):
    """Conta tags distintas (uma vez por arquivo) e também a versão 'stripped' sem prefixos do Ray."""
    raw = Counter()
    stripped = Counter()
    for path in event_files:
        seen_raw = set()
        seen_stripped = set()
        steps_seen = 0
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                ts = strip_ray_prefix(t)
                if t not in seen_raw:
                    raw[t] += 1; seen_raw.add(t)
                if ts not in seen_stripped:
                    stripped[ts] += 1; seen_stripped.add(ts)
            steps_seen += 1
            if steps_seen >= max_per_file:
                break
    return raw, stripped

# Tags comuns do RLlib para recompensa/retorno (forma "stripped")
REWARD_TAG_PREFERENCE = [
    "evaluation/episode_reward_mean",
    "episode_reward_mean",
    "rollout/episode_reward_mean",
    "episode_return_mean",
    "evaluation/episode_return_mean",
    "train/episode_reward_mean",
    "train/mean_reward",
    "charts/episode_reward_mean",
]
REWARD_REGEX = re.compile(
    r"(episode|ep).*(reward|return).*(mean|avg)|"
    r"(reward|return).*(episode|ep).*(mean|avg)",
    re.IGNORECASE,
)

# Prioridades para eixo X (forma "stripped")
X_AXIS_TAG_PREFERENCE = [
    "timesteps_total",
    "env_steps_sampled",
    "env_steps_trained",
    "training_iteration",
    "episodes_total",
]

def choose_reward_tag(available_stripped_tags: Set[str]) -> Optional[str]:
    for t in REWARD_TAG_PREFERENCE:
        if t in available_stripped_tags:
            return t
    candidates = [t for t in available_stripped_tags if REWARD_REGEX.search(t)]
    if candidates:
        candidates.sort(key=lambda s: (len(s), s))
        return candidates[0]
    return None

def choose_x_axis_tag(available_stripped_tags: Set[str]) -> Optional[str]:
    for t in X_AXIS_TAG_PREFERENCE:
        if t in available_stripped_tags:
            return t
    return None  # fallback: passo interno do TensorBoard

# ---------- extração ----------
def extract_series(event_paths, y_tag, x_tag=None):
    """
    Agrega um escalar y_tag em múltiplas execuções.
    Se x_tag for fornecida, alinha usando o 'step' do TF e retorna x a partir dessa tag; senão usa 'step'.
    Retorna (xs_sorted, mean_y, std_y).
    As tags devem ser fornecidas em forma *stripped* (sem prefixo ray/*),
    mas o matching aceita ambas as formas.
    """
    def matches(tag, wanted):
        return strip_ray_prefix(tag) == wanted

    x_by_step_all_runs = defaultdict(list) if x_tag else None
    y_by_step = defaultdict(list)

    for path in event_paths:
        x_by_step = {}       # step -> x
        y_by_step_run = {}   # step -> y
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                if x_tag and matches(t, x_tag):
                    x_by_step[e.step] = float(v.simple_value)
                if matches(t, y_tag):
                    y_by_step_run[e.step] = float(v.simple_value)
        if x_tag:
            for s, xv in x_by_step.items():
                x_by_step_all_runs[s].append(xv)
        for s, yv in y_by_step_run.items():
            y_by_step[s].append(yv)

    steps = sorted(y_by_step.keys())
    if not steps:
        return [], [], []
    if x_tag:
        xs = [np.mean(x_by_step_all_runs[s]) if s in x_by_step_all_runs else float(s) for s in steps]
    else:
        xs = [float(s) for s in steps]

    means = [np.mean(y_by_step[s]) for s in steps]
    stds  = [np.std(y_by_step[s])  for s in steps]
    order = np.argsort(xs)
    xs = list(np.array(xs)[order])
    means = list(np.array(means)[order])
    stds = list(np.array(stds)[order])
    return xs, means, stds

# ---------- agrupar por algoritmo ----------
def infer_algorithm(dirname: str) -> str:
    """
    Obtém o nome do algoritmo pelo prefixo antes do primeiro '_' no nome da pasta.
    Ex.: 'happo_mlp_sunt_bus_...' -> 'happo'
    """
    base = os.path.basename(dirname).lower()
    if "_" in base:
        return base.split("_", 1)[0]
    return base  # fallback: a pasta inteira

def group_folders_by_algorithm(base_dir: str) -> Dict[str, List[str]]:
    """Agrupa subpastas imediatas por algoritmo."""
    groups = defaultdict(list)
    for d in os.listdir(base_dir):
        full = os.path.join(base_dir, d)
        if os.path.isdir(full):
            algo = infer_algorithm(d)
            groups[algo].append(full)
    # Se houver events diretamente no base_dir, trate como pasta do próprio nome do base_dir
    root_events = get_event_files_in_dir(base_dir)
    if root_events:
        algo = infer_algorithm(os.path.basename(base_dir))
        groups[algo].append(base_dir)
    return groups

# ---------- plotagem ----------
def plot_smoothed_line(xs, means, stds, label="metric", title="", normalize=False, sigma=3, save_path=None):
    plt.figure(figsize=(11, 7))
    means = np.array(means); stds = np.array(stds); xs = np.array(xs)
    # Suaviza só se houver pontos suficientes
    sm_m = gaussian_filter1d(means, sigma=sigma) if len(means) >= 3 else means
    sm_s = gaussian_filter1d(stds,  sigma=sigma) if len(stds)  >= 3 else stds
    if normalize:
        mn, mx = sm_m.min(), sm_m.max()
        rng = (mx - mn) if mx > mn else 1.0
        sm_m = (sm_m - mn) / rng
        sm_s = sm_s / rng
        lower = np.clip(sm_m - sm_s, 0, 1); upper = np.clip(sm_m + sm_s, 0, 1)
        plt.ylabel("Normalized value")
    else:
        lower, upper = sm_m - sm_s, sm_m + sm_s
        plt.ylabel("Value")
    plt.plot(xs, sm_m, label=label)
    plt.fill_between(xs, lower, upper, alpha=0.2)
    xlabel = "Timesteps"
    if not len(xs) or (xs[0] == 0 and max(xs, default=0) <= len(xs)):
        xlabel = "Step"
    plt.xlabel(xlabel)
    plt.title(title, fontsize=16, fontweight="bold")  # nome do algoritmo em destaque no topo
    plt.legend(); plt.grid(True); plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=160)
        print(f"💾 Salvo: {save_path}")
        plt.close()
    else:
        plt.show()

# ---------- driver: uma figura por ALGORITMO ----------
if __name__ == "__main__":
    # >>> ALTERE AQUI <<<
    # Ex.: base_dir = "/mnt/ssd1/rafael/graph-exploration/exp_results_copy"
    base_dir = "./exp_results_copy"

    # Agrupa subpastas por algoritmo
    algo_groups = group_folders_by_algorithm(base_dir)

    charts_dir = os.path.join(base_dir, "charts")
    os.makedirs(charts_dir, exist_ok=True)

    for algo, folders in sorted(algo_groups.items()):
        # Junta todos os event files de TODAS as pastas desse algoritmo
        event_files = []
        for f in folders:
            event_files.extend(get_event_files_in_dir(f))
        event_files = sorted(set(event_files))
        if not event_files:
            print(f"… {algo}: sem arquivos de eventos.")
            continue

        # Descobre tags (uma por algoritmo, agregando todos os runs/pastas do grupo)
        _, stripped_freq = tag_frequency(event_files, max_per_file=800)
        available = set(stripped_freq.keys())
        y_tag = choose_reward_tag(available)
        x_tag = choose_x_axis_tag(available)

        if y_tag is None:
            print(f"⚠️  {algo}: não encontrei tag de recompensa/retorno. Ignorando.")
            continue

        xs, means, stds = extract_series(event_files, y_tag=y_tag, x_tag=x_tag)
        if not xs:
            print(f"⚠️  {algo}: sem dados para '{y_tag}'. Ignorando.")
            continue

        # Título com nome do algoritmo (no topo) + info de eixos
        title = f"{algo.upper()} — {y_tag}" + (f" vs {x_tag}" if x_tag else "")
        fn = f"{algo}__{y_tag.replace('/','_')}" + (f"__{x_tag.replace('/','_')}" if x_tag else "__step") + ".png"
        save_path = os.path.join(charts_dir, fn)

        plot_smoothed_line(
            xs, means, stds,
            label=y_tag,
            title=title,
            normalize=False,
            sigma=3,
            save_path=save_path
        )

    print("✅ Concluído.")


… charts: sem arquivos de eventos.
💾 Salvo: ./exp_results_copy/charts/coma__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/exp__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/happo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/hatrpo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/ippo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/itrpo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/maa2c__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/mappo__episode_reward_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/matrpo__episode_reward_mean__timesteps_total.png
✅ Concluído.


In [4]:
# -*- coding: utf-8 -*-
# --- Plots por ALGORITMO para métricas específicas ---
# Gera: episode_len_mean e (num único gráfico) os total_loss de policy_0..policy_4

import os
import re
import glob
from collections import defaultdict, Counter
from typing import Optional, Set, Dict, List

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.python.summary.summary_iterator import summary_iterator
from scipy.ndimage import gaussian_filter1d

# ---------- leitura de eventos ----------
def get_event_files_in_dir(d: str) -> List[str]:
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))

def safe_iter_events(path: str):
    try:
        for e in summary_iterator(path):
            yield e
    except Exception as ex:
        print(f"⚠️  Ignorando {path} (erro de leitura: {ex})")

# ---------- utilitários de tags ----------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")
def strip_ray_prefix(tag: str) -> str:
    # remove "ray/tune/" ou "ray/train/" etc. para facilitar matching
    return RAY_PREFIX.sub("", tag)

def tag_frequency(event_files, max_per_file=500):
    """Conta tags distintas (uma vez por arquivo) e também a versão 'stripped' sem prefixos do Ray."""
    raw = Counter()
    stripped = Counter()
    for path in event_files:
        seen_raw = set()
        seen_stripped = set()
        steps_seen = 0
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                ts = strip_ray_prefix(t)
                if t not in seen_raw:
                    raw[t] += 1; seen_raw.add(t)
                if ts not in seen_stripped:
                    stripped[ts] += 1; seen_stripped.add(ts)
            steps_seen += 1
            if steps_seen >= max_per_file:
                break
    return raw, stripped

# Prioridades para eixo X (forma "stripped")
X_AXIS_TAG_PREFERENCE = [
    "timesteps_total",
    "env_steps_sampled",
    "env_steps_trained",
    "training_iteration",
    "episodes_total",
]

def choose_x_axis_tag(available_stripped_tags: Set[str]) -> Optional[str]:
    for t in X_AXIS_TAG_PREFERENCE:
        if t in available_stripped_tags:
            return t
    return None  # fallback: passo interno do TensorBoard

# ---------- extração ----------
def extract_series(event_paths, y_tag, x_tag=None):
    """
    Agrega um escalar y_tag em múltiplas execuções.
    Se x_tag for fornecida, alinha usando o 'step' do TF e retorna x a partir dessa tag; senão usa 'step'.
    Retorna (xs_sorted, mean_y, std_y).
    As tags devem ser fornecidas em forma *stripped* (sem prefixo ray/*),
    mas o matching aceita ambas as formas.
    """
    def matches(tag, wanted):
        return strip_ray_prefix(tag) == wanted

    x_by_step_all_runs = defaultdict(list) if x_tag else None
    y_by_step = defaultdict(list)

    for path in event_paths:
        x_by_step = {}       # step -> x
        y_by_step_run = {}   # step -> y
        for e in safe_iter_events(path):
            for v in e.summary.value:
                t = v.tag
                if x_tag and matches(t, x_tag):
                    x_by_step[e.step] = float(v.simple_value)
                if matches(t, y_tag):
                    y_by_step_run[e.step] = float(v.simple_value)
        if x_tag:
            for s, xv in x_by_step.items():
                x_by_step_all_runs[s].append(xv)
        for s, yv in y_by_step_run.items():
            y_by_step[s].append(yv)

    steps = sorted(y_by_step.keys())
    if not steps:
        return [], [], []
    if x_tag:
        xs = [np.mean(x_by_step_all_runs[s]) if s in x_by_step_all_runs else float(s) for s in steps]
    else:
        xs = [float(s) for s in steps]

    means = [np.mean(y_by_step[s]) for s in steps]
    stds  = [np.std(y_by_step[s])  for s in steps]
    order = np.argsort(xs)
    xs = list(np.array(xs)[order])
    means = list(np.array(means)[order])
    stds = list(np.array(stds)[order])
    return xs, means, stds

# ---------- plotagem ----------
def plot_smoothed_line(xs, means, stds, label="metric", title="", normalize=False, sigma=3, save_path=None):
    plt.figure(figsize=(11, 7))
    means = np.array(means); stds = np.array(stds); xs = np.array(xs)
    # Suaviza só se houver pontos suficientes
    sm_m = gaussian_filter1d(means, sigma=sigma) if len(means) >= 3 else means
    sm_s = gaussian_filter1d(stds,  sigma=sigma) if len(stds)  >= 3 else stds
    if normalize:
        mn, mx = sm_m.min(), sm_m.max()
        rng = (mx - mn) if mx > mn else 1.0
        sm_m = (sm_m - mn) / rng
        sm_s = sm_s / rng
        lower = np.clip(sm_m - sm_s, 0, 1); upper = np.clip(sm_m + sm_s, 0, 1)
        plt.ylabel("Normalized value")
    else:
        lower, upper = sm_m - sm_s, sm_m + sm_s
        plt.ylabel("Value")
    plt.plot(xs, sm_m, label=label)
    plt.fill_between(xs, lower, upper, alpha=0.2)
    xlabel = "Timesteps"
    if not len(xs) or (xs[0] == 0 and max(xs, default=0) <= len(xs)):
        xlabel = "Step"
    plt.xlabel(xlabel)
    plt.title(title, fontsize=16, fontweight="bold")
    plt.legend(); plt.grid(True); plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=160)
        print(f"💾 Salvo: {save_path}")
        plt.close()
    else:
        plt.show()

def plot_multi_smoothed(series_dict, title="", normalize=False, sigma=3, save_path=None):
    """
    series_dict: {label: (xs, means, stds)}
    Plota múltiplas séries (mesmo eixo X aproximado).
    """
    plt.figure(figsize=(11, 7))
    xlabel = "Timesteps"
    any_x = None

    for label, (xs, means, stds) in series_dict.items():
        if not xs:
            continue
        any_x = xs if any_x is None else any_x
        means = np.array(means); stds = np.array(stds); xs = np.array(xs)
        sm_m = gaussian_filter1d(means, sigma=sigma) if len(means) >= 3 else means
        sm_s = gaussian_filter1d(stds,  sigma=sigma) if len(stds)  >= 3 else stds
        if normalize:
            mn, mx = sm_m.min(), sm_m.max()
            rng = (mx - mn) if mx > mn else 1.0
            sm_m = (sm_m - mn) / rng
            sm_s = sm_s / rng
            lower = np.clip(sm_m - sm_s, 0, 1); upper = np.clip(sm_m + sm_s, 0, 1)
            plt.ylabel("Normalized value")
        else:
            lower, upper = sm_m - sm_s, sm_m + sm_s
            plt.ylabel("Value")
        plt.plot(xs, sm_m, label=label)
        plt.fill_between(xs, lower, upper, alpha=0.15)

    if any_x:
        if any_x[0] == 0 and max(any_x, default=0) <= len(any_x):
            xlabel = "Step"
    plt.xlabel(xlabel)
    plt.title(title, fontsize=16, fontweight="bold")
    plt.legend(); plt.grid(True); plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=160)
        print(f"💾 Salvo: {save_path}")
        plt.close()
    else:
        plt.show()

# ---------- agrupar por algoritmo ----------
def infer_algorithm(dirname: str) -> str:
    """
    Obtém o nome do algoritmo pelo prefixo antes do primeiro '_' no nome da pasta.
    Ex.: 'happo_mlp_sunt_bus_...' -> 'happo'
    """
    base = os.path.basename(dirname).lower()
    if "_" in base:
        return base.split("_", 1)[0]
    return base  # fallback: a pasta inteira

def group_folders_by_algorithm(base_dir: str) -> Dict[str, List[str]]:
    """Agrupa subpastas imediatas por algoritmo."""
    groups = defaultdict(list)
    for d in os.listdir(base_dir):
        full = os.path.join(base_dir, d)
        if os.path.isdir(full):
            algo = infer_algorithm(d)
            groups[algo].append(full)
    # Se houver events diretamente no base_dir, trate como pasta do próprio nome do base_dir
    root_events = get_event_files_in_dir(base_dir)
    if root_events:
        algo = infer_algorithm(os.path.basename(base_dir))
        groups[algo].append(base_dir)
    return groups

# ---------- métricas desejadas (forma 'stripped') ----------
# Originais com prefixo:
# - ray/tune/episode_len_mean
# - ray/tune/info/learner/policy_k/learner_stats/total_loss  (k=0..4)
EPISODE_LEN_TAG = "episode_len_mean"
POLICY_LOSS_TAGS = [f"info/learner/policy_{k}/learner_stats/total_loss" for k in range(5)]

# ---------- driver ----------
if __name__ == "__main__":
    # >>> ALTERE AQUI <<<
    # Ex.: base_dir = "/mnt/ssd1/rafael/graph-exploration/exp_results_copy"
    base_dir = "./exp_results_copy"

    # Agrupa subpastas por algoritmo
    algo_groups = group_folders_by_algorithm(base_dir)

    charts_dir = os.path.join(base_dir, "charts")
    os.makedirs(charts_dir, exist_ok=True)

    for algo, folders in sorted(algo_groups.items()):
        # Junta todos os event files de TODAS as pastas desse algoritmo
        event_files = []
        for f in folders:
            event_files.extend(get_event_files_in_dir(f))
        event_files = sorted(set(event_files))
        if not event_files:
            print(f"… {algo}: sem arquivos de eventos.")
            continue

        # Descobre tags (uma por algoritmo, agregando todos os runs/pastas do grupo)
        _, stripped_freq = tag_frequency(event_files, max_per_file=800)
        available = set(stripped_freq.keys())
        x_tag = choose_x_axis_tag(available)

        # 1) episode_len_mean (se existir)
        if EPISODE_LEN_TAG in available:
            xs, means, stds = extract_series(event_files, y_tag=EPISODE_LEN_TAG, x_tag=x_tag)
            if xs:
                title = f"{algo.upper()} — {EPISODE_LEN_TAG}" + (f" vs {x_tag}" if x_tag else "")
                fn = f"{algo}/{algo}__{EPISODE_LEN_TAG.replace('/','_')}" + (f"__{x_tag.replace('/','_')}" if x_tag else "__step") + ".png"
                save_path = os.path.join(charts_dir, fn)
                plot_smoothed_line(
                    xs, means, stds,
                    label=EPISODE_LEN_TAG,
                    title=title,
                    normalize=False,
                    sigma=3,
                    save_path=save_path
                )
            else:
                print(f"⚠️  {algo}: sem dados para '{EPISODE_LEN_TAG}'.")
        else:
            print(f"… {algo}: tag '{EPISODE_LEN_TAG}' não encontrada.")

        # 2) total_loss das policies (plota todas no mesmo gráfico, somente as que existirem)
        series = {}
        present_losses = [t for t in POLICY_LOSS_TAGS if t in available]
        if not present_losses:
            print(f"… {algo}: nenhuma policy loss encontrada (policy_0..4).")
        else:
            for t in present_losses:
                xs, means, stds = extract_series(event_files, y_tag=t, x_tag=x_tag)
                if xs:
                    series[t.split('/')[2]] = (xs, means, stds)  # label curto: policy_k
            if series:
                title = f"{algo.upper()} — learner_stats/total_loss por policy" + (f" vs {x_tag}" if x_tag else "")
                fn = f"{algo}/{algo}__policies_total_loss" + (f"__{x_tag.replace('/','_')}" if x_tag else "__step") + ".png"
                save_path = os.path.join(charts_dir, fn)
                plot_multi_smoothed(series, title=title, normalize=False, sigma=3, save_path=save_path)
            else:
                print(f"⚠️  {algo}: sem dados válidos para total_loss das policies.")

    print("✅ Concluído.")


… charts: sem arquivos de eventos.
💾 Salvo: ./exp_results_copy/charts/coma/coma__episode_len_mean__timesteps_total.png
… coma: nenhuma policy loss encontrada (policy_0..4).
💾 Salvo: ./exp_results_copy/charts/exp/exp__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/exp/exp__policies_total_loss__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/happo/happo__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/happo/happo__policies_total_loss__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/hatrpo/hatrpo__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/hatrpo/hatrpo__policies_total_loss__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/ippo/ippo__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/ippo/ippo__policies_total_loss__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/itrpo/itrpo__episode_len_mean__timesteps_total.png
💾 Salvo: ./exp_results_copy/charts/itrpo/itrpo__p